# Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `games/`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [19]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [20]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [21]:
# Make sure your .env file contains:
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [22]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)

### VectorDB Instance

In [23]:
# Instantiate a persistent ChromaDB client
# Data will be saved to the 'chromadb' directory so it survives restarts
chroma_client = chromadb.PersistentClient(path="chromadb")

print(f"ChromaDB client ready. Existing collections: {chroma_client.list_collections()}")

ChromaDB client ready. Existing collections: [Collection(name=udaplay)]


### Collection

In [24]:
# Use OpenAI's text-embedding-ada-002 model to generate embeddings.
# This same function must be used when loading the collection later (Part 02).
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    model_name="text-embedding-ada-002"
)

print("Embedding function ready.")

Embedding function ready.


In [25]:
# Create (or re-create) the 'udaplay' collection.
# get_or_create_collection is idempotent: safe to run multiple times.
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # cosine distance works well for semantic search
)

print(f"Collection '{collection.name}' ready. Documents stored: {collection.count()}")

Collection 'udaplay' ready. Documents stored: 15


### Add documents

In [26]:
# Make sure you have a directory "games/" containing .json game files
data_dir = "games"

added = 0
skipped = 0

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Build a rich text representation of the game to embed.
    # Including platform, name, year, genre, publisher, and description
    # gives the embedder full context for semantic search.
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) "
        f"- Genre: {game.get('Genre', 'Unknown')} "
        f"- Publisher: {game.get('Publisher', 'Unknown')} "
        f"- {game['Description']}"
    )

    # Use the file name (without extension, e.g. '001') as the document ID.
    # This makes IDs stable and predictable across re-runs.
    doc_id = os.path.splitext(file_name)[0]

    # Check if already present to avoid duplicate-key errors on re-runs
    existing = collection.get(ids=[doc_id])
    if existing["ids"]:
        skipped += 1
        continue

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]   # Store full game dict as metadata for easy retrieval
    )
    added += 1

print(f"Done! Added: {added} | Skipped (already present): {skipped}")
print(f"Total documents in collection: {collection.count()}")

Done! Added: 0 | Skipped (already present): 15
Total documents in collection: 15


### Verify the Collection

In [27]:
# Quick sanity check: run a test query to confirm embeddings + retrieval work
test_results = collection.query(
    query_texts=["3D platformer game with a famous plumber"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print("Test query: '3D platformer game with a famous plumber'")
print("-" * 60)
for doc, meta, dist in zip(
    test_results["documents"][0],
    test_results["metadatas"][0],
    test_results["distances"][0]
):
    print(f"  Game   : {meta.get('Name')} ({meta.get('Platform')}, {meta.get('YearOfRelease')})")
    print(f"  Score  : {1 - dist:.4f} (cosine similarity)")
    print(f"  Excerpt: {doc[:120]}...")
    print()

Test query: '3D platformer game with a famous plumber'
------------------------------------------------------------
  Game   : Super Mario 64 (Nintendo 64, 1996)
  Score  : 0.8519 (cosine similarity)
  Excerpt: [Nintendo 64] Super Mario 64 (1996) - Genre: Platformer - Publisher: Nintendo - A groundbreaking 3D platformer that set ...

  Game   : Super Mario World (Super Nintendo Entertainment System (SNES), 1990)
  Score  : 0.8338 (cosine similarity)
  Excerpt: [Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - Genre: Platformer - Publisher: Nintendo - A clas...

  Game   : Marvel's Spider-Man (PlayStation 4, 2018)
  Score  : 0.7956 (cosine similarity)
  Excerpt: [PlayStation 4] Marvel's Spider-Man (2018) - Genre: Action-adventure - Publisher: Sony Interactive Entertainment - An op...

